In [68]:
# Importing Libraries
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model  import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from xgboost import XGBClassifier

In [69]:
# Loading Data
df = pd.read_excel('/content/chronic_kidney_disease.xlsx')

In [70]:
df.head()

,'age','bp','sg','al','su','rbc','pc','pcc','ba','bgr',...,'pcv','wbcc','rbcc','htn','dm','cad','appet','pe','ane','class'
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [71]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace("'", "", regex=False)
    .str.replace('"', "", regex=False)
    .str.lower()
)

In [72]:
df.rename(columns={
    "age": "Age_years",
    "bp": "Blood_Pressure_mmHg",
    "sg": "Specific_Gravity",
    "al": "Urine_Albumin",
    "su": "Urine_Sugar",
    "rbc": "Urine_RBC",
    "pc": "Urine_Pus_Cell",
    "bgr": "Random_Blood_Glucose_mg_dL",
    "bu": "Blood_Urea_mg_dL",
    "sc": "Serum_Creatinine_mg_dL",
    "sod": "Sodium_mEq_L",
    "pot": "Potassium_mEq_L",
    "hemo": "Hemoglobin_g_dL",
    "pcv": "Packed_Cell_Volume",
    "wbcc": "White_Blood_Cell_Count",
    "rbcc": "Red_Blood_Cell_Count",
    "class": "CKD_Status"
}, inplace=True)

In [73]:
df.head()

,Age_years,Blood_Pressure_mmHg,Specific_Gravity,Urine_Albumin,Urine_Sugar,Urine_RBC,Urine_Pus_Cell,pcc,ba,Random_Blood_Glucose_mg_dL,...,Packed_Cell_Volume,White_Blood_Cell_Count,Red_Blood_Cell_Count,htn,dm,cad,appet,pe,ane,CKD_Status
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [74]:
# Selected Feautures
selected_features = [
    "Age_years",
    "Blood_Pressure_mmHg",
    "Specific_Gravity",
    "Urine_Albumin",
    "Urine_Sugar",
    "Urine_RBC",
    "Urine_Pus_Cell",
    "Random_Blood_Glucose_mg_dL",
    "Blood_Urea_mg_dL",
    "Serum_Creatinine_mg_dL",
    "Sodium_mEq_L",
    "Potassium_mEq_L",
    "Hemoglobin_g_dL",
    "Packed_Cell_Volume",
    "White_Blood_Cell_Count",
    "Red_Blood_Cell_Count",
    "CKD_Status"
]

In [75]:

df = df[selected_features]

In [76]:
# Cleaning Data
df.replace("?" , np.nan , inplace = True)
df.replace("",np.nan,inplace=True)

for col in df.select_dtypes(include="object").columns:
  df[col]=  df[col].astype(str).str.strip().str.lower()


In [77]:
# Fixing messy yes/no
df.replace({
    "\tyes": "yes",
    " yes": "yes",
    "no ": "no",
    "\tno": "no"
}, inplace=True)

In [78]:
# Conert Numeric Columns
numeric_columns = [
    "Age_years",
    "Blood_Pressure_mmHg",
    "Random_Blood_Glucose_mg_dL",
    "Blood_Urea_mg_dL",
    "Serum_Creatinine_mg_dL",
    "Sodium_mEq_L",
    "Potassium_mEq_L",
    "Hemoglobin_g_dL",
    "Packed_Cell_Volume",
    "White_Blood_Cell_Count",
    "Red_Blood_Cell_Count"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [79]:
# Remove Impossible Values
df.loc[df["Age_years"] <= 0, "Age_years"] = np.nan
df.loc[df["Blood_Pressure_mmHg"] <= 0, "Blood_Pressure_mmHg"] = np.nan

In [80]:
# Encode Target
df["CKD_Status"] = df["CKD_Status"].map({
    "ckd": 1,
    "notckd": 0
})

In [81]:
# Normalize target first
df["CKD_Status"] = (
    df["CKD_Status"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Replace missing markers
df["CKD_Status"].replace(["?", "", "nan"], np.nan, inplace=True)

/tmp/ipython-input-1351/2546292446.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["CKD_Status"].replace(["?", "", "nan"], np.nan, inplace=True)


In [82]:
df = df[df["CKD_Status"].notna()]

In [83]:
print("Missing in target:", df["CKD_Status"].isna().sum())

Missing in target: 0


In [84]:
# Split Feautures
X = df.drop("CKD_Status" , axis=1)
y=df["CKD_Status"]

In [85]:
X

,Age_years,Blood_Pressure_mmHg,Specific_Gravity,Urine_Albumin,Urine_Sugar,Urine_RBC,Urine_Pus_Cell,Random_Blood_Glucose_mg_dL,Blood_Urea_mg_dL,Serum_Creatinine_mg_dL,Sodium_mEq_L,Potassium_mEq_L,Hemoglobin_g_dL,Packed_Cell_Volume,White_Blood_Cell_Count,Red_Blood_Cell_Count
0,48.0,80.0,1.020,1.0,0.0,nan,normal,121.0,36.0,1.2,NaN,NaN,15.4,44.0,7800.0,5.2
1,7.0,50.0,1.020,4.0,0.0,nan,normal,NaN,18.0,0.8,NaN,NaN,11.3,38.0,6000.0,NaN
2,62.0,80.0,1.010,2.0,3.0,normal,normal,423.0,53.0,1.8,NaN,NaN,9.6,31.0,7500.0,NaN
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,117.0,56.0,3.8,111.0,2.5,11.2,32.0,6700.0,3.9
4,51.0,80.0,1.010,2.0,0.0,normal,normal,106.0,26.0,1.4,NaN,NaN,11.6,35.0,7300.0,4.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,55.0,80.0,1.020,0.0,0.0,normal,normal,140.0,49.0,0.5,150.0,4.9,15.7,47.0,6700.0,4.9
396,42.0,70.0,1.025,0.0,0.0,normal,normal,75.0,31.0,1.2,141.0,3.5,16.5,54.0,7800.0,6.2
397,12.0,80.0,1.020,0.0,0.0,normal,normal,100.0,26.0,0.6,137.0,4.4,15.8,49.0,6600.0,5.4
398,17.0,60.0,1.025,0.0,0.0,normal,normal,114.0,50.0,1.0,135.0,4.9,14.2,51.0,7200.0,5.9


In [86]:
y

,CKD_Status
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0
...,...
395,0.0
396,0.0
397,0.0
398,0.0


In [87]:

categorical_columns = X.select_dtypes(include="object").columns.tolist()

In [88]:
# Pre Processing
num_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numeric_columns),
    ("cat", cat_pipeline, categorical_columns)
])



In [89]:
# Stcked Model

base_models=[
    ("lr",LogisticRegression(max_iter=1000)),
    ("rf", RandomForestClassifier(n_estimators=200,random_state=42)),
    ("xgb" , XGBClassifier(n_estimators = 200 , eval_metric="logloss" , random_state=42)),
    ("svm"  ,  SVC(probability=True))
]

stack_model = StackingClassifier(
    estimators = base_models,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessing",preprocessor),
    ("model",stack_model)
])

In [90]:
y

,CKD_Status
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0
...,...
395,0.0
396,0.0
397,0.0
398,0.0


In [91]:
# Train & Evaluate
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nCross Validation Accuracy:",
      cross_val_score(pipeline, X, y, cv=5).mean())



Accuracy: 0.9375
ROC-AUC: 0.9906666666666667

Cross Validation Accuracy: 0.9549683544303799


In [92]:
# -------------------------------------------
# 11. SAVE MODEL
# -------------------------------------------
joblib.dump(pipeline, "ckd_reliable_cleaned_model.pkl")
print("\nModel saved successfully.")


Model saved successfully.
